# 🧪 综合实验 18：大规模 RAG 系统构建与端到端评测

本实验是 **检索增强生成 (RAG, Retrieval-Augmented Generation)** 章节的综合实践。我们将围绕大型中文小说语料库（`data/knowledge.txt`，~755KB，含9356行《剑来》第一章小说内容），构建一个完整的、工业级的本地 RAG 系统。

本实验是一个**验证性实验**，内容上已为您提供完整且高度优化的算法框架。您需要通过运行代码，观察各项指标与图表变化，深入探究各组件在 RAG 中的关键角色与性能折中。

## 🎯 实验核心内容

1. **🧱 任务一：文本切分艺术（Chunking）与持久化向量检索**
   - 比较 **策略 A（Naive 固定切片，size=100）** 与 **策略 B（Overlap 滑动重叠切片，size=150, overlap=30）** 的检索 **Recall@3** 差异。
   - 学习向量生成与基于 PyTorch 的本地**索引持久化 (`torch.save/load`)** 实战。
2. **⚖️ 任务二：两阶段检索消融实验——深度重排（Re-ranking）**
   - 消融对比：**纯向量粗排 Top-3** vs. **向量粗筛 Top-10 + `bge-reranker-v2-m3` 重排 Top-3**。
   - 分析检索精度与系统时延（Latency）的双轴折中（Trade-off）。
3. **🧠 任务三：前置问题改写（Query Rewriting）与 HyDE 召回价值分析**
   - **指代模糊问题专项评测**：专门针对包含“那丫头”、“那小子”、“半路师傅”等极易脱靶的口语化提问，横向对比 **直搜**、**大模型前置改写 (Rewrite)** 以及 **假想文档检索 (HyDE)** 的 Recall@3 飞跃。
4. **💻 任务四：端到端全维学术消融实验 (End-to-End Ablation Study)**
   - 统一 Benchmark 数据集下评估四路方案：**原始模型裸答**、**全文本无脑注入提示词（截断CPU压力测试）**、**Naive RAG** 以及 **Advanced RAG (改写 + 重叠分块 + 重排)** 的端到端 Recall 和时延表现。
5. **🔖 任务五：原文引用（Attribution）与安全拒答边界判定**
   - 实现工业级带 `[引用依据]` 段落编号的 RAG 回答格式化机制。
   - 在 10 道越界安全题型上考核系统的安全拒答率，以起到防幻觉防御的作用。


---
## 🛠️ 环境初始化

本节导入 RAG 系统所需要的核心依赖库，并配置精美的数据科学可视化风格与中文字体显示环境。

In [ ]:
import os
import json
import time
import warnings
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm
from huggingface_hub import snapshot_download
from transformers import (
    AutoTokenizer, 
    AutoModelForCausalLM, 
    AutoModel, 
    AutoModelForSequenceClassification
)

# 过滤无关警告
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'

# 设备检测逻辑：优先支持 GPU (CUDA)、Mac (MPS)，其次 fallback 至 CPU
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f'📌 当前系统检测并启用可用计算设备: {device.upper()}')

# 设置 Matplotlib 中文显示环境
target_fonts = ['SimHei', 'Arial Unicode MS', 'Microsoft YaHei', 'DejaVu Sans']
available_fonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
for font in target_fonts:
    if font in available_fonts:
        plt.rcParams['font.sans-serif'] = [font]
        break
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.sans-serif'][0])

# 确保输出文件夹存在
os.makedirs('outputs', exist_ok=True)

# 在最开始载入黄金问答评测集，避免后续单元格顺序导致未声明错误
benchmark_path = 'data/benchmark_dataset.json'
if os.path.exists(benchmark_path):
    with open(benchmark_path, 'r', encoding='utf-8') as f:
        BENCHMARK_DATASET = json.load(f)
    print(f'📬 黄金评测集已成功从 {benchmark_path} 加载完毕！')
else:
    BENCHMARK_DATASET = []
    print(f'⚠️ 警告：未在 {benchmark_path} 找到评测集 JSON 文件，请确保文件已放置在 data 文件夹下。')

# ==================== 🤖 云端大模型 API 载入与自适应调用 ====================
from openai import OpenAI

# ============================================================
# 👇 把你的 API Key 填在下面的引号里（通义千问或硅基流动二选一即可）
# ============================================================
DASH_SCOPE_API_KEY = "" # 例如: "sk-abc123..."
SILICONFLOW_API_KEY = "" # 例如: "sk-abc123..."

openai_client = None
CLOUD_MODEL_NAME = ""

if DASH_SCOPE_API_KEY:
    openai_client = OpenAI(
        api_key=DASH_SCOPE_API_KEY,
        base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    )
    CLOUD_MODEL_NAME = "qwen3.5-flash"
elif SILICONFLOW_API_KEY:
    openai_client = OpenAI(
        api_key=SILICONFLOW_API_KEY,
        base_url="https://api.siliconflow.cn/v1",
    )
    CLOUD_MODEL_NAME = "deepseek-ai/DeepSeek-V3.2"

# 如果学生没有配置，输出友好指引提示，但绝不卡死运行
if not openai_client:
    print('📢 [云端评测提示] 未检测到 DASH_SCOPE_API_KEY 或 SILICONFLOW_API_KEY。')
    print('   云端智能裁判打分 (LLM-as-a-judge) 将以本地 Mock 语义匹配对齐机制优雅降级运行。')
    print('   如需启用真实云端模型，请在下方直接补充赋值：')
    print('   >>> DASH_SCOPE_API_KEY = "你的通义千问API Key" 或 SILICONFLOW_API_KEY = "你的硅基流动API Key"')
else:
    print(f'✨ 成功加载云端 API 密钥！已实例化 OpenAI 客户端，模型: {CLOUD_MODEL_NAME}，将在消融实验中自动激活云端裁判打分！')

def call_cloud_llm(prompt, system_prompt='你是一个专业严谨的评测裁判。'):
    '''通用云端大模型接口，基于 OpenAI SDK 兼容格式调用，支持超时保护与优雅错误处理'''
    global openai_client, CLOUD_MODEL_NAME
    if not openai_client:
        return None
    try:
        response = openai_client.chat.completions.create(
            model=CLOUD_MODEL_NAME,
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': prompt}
            ],
            timeout=10,
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f'[云端模型调用故障: {e}]'
# =========================================================================

print('✅ 所有核心数据科学依赖库导入成功，并且 outputs/ 文件夹已就绪！')

### 🤖 加载 RAG 全链路“三剑客”模型

我们将从本地 `models/` 目录加载 RAG 全链路所需的三个模型：
1. **大语言模型 (LLM)**：`models/Qwen2-0.5B-Instruct` — 用于回答生成、问题改写以及 HyDE 假想文档生成。
2. **文本向量化模型 (Embedding)**：`models/bge-small-zh-v1.5` — 用于文本切片编码和语义相似度计算。
3. **文本重排模型 (Reranker)**：`models/bge-reranker-v2-m3` — 用于对粗排召回的结果进行二次交叉注意力打分。

我们编写带“自愈性完整校验”的自适应模型加载函数，自动适应 CPU 环境并校验文件完整度。

In [ ]:
def load_rag_models():
    '''加载 RAG 全流程所需的本地模型，包含 snapshot 检查确保文件完整'''
    models_root = Path('models')
    llm_id = 'Qwen/Qwen2-0.5B-Instruct'
    emb_id = 'BAAI/bge-small-zh-v1.5'
    rerank_id = 'BAAI/bge-reranker-v2-m3'
    
    def quick_load(repo_id, model_type='llm'):
        model_name = repo_id.split('/')[-1]
        local_path = models_root / model_name
        print(f'📦 正在校验并载入本地模型: {model_name}...')
        
        # 物理检查并补充下载缺失权重
        try:
            snapshot_download(
                repo_id=repo_id, 
                local_dir=str(local_path),
                local_dir_use_symlinks=False,
                ignore_patterns=['*.msgpack', '*.h5', '*.ot']
            )
        except Exception as e:
            print(f'⚠️ 无法连接在线镜像源/代理异常 ({e})，将尝试直接使用本地已缓存的模型文件...')
        
        tokenizer = AutoTokenizer.from_pretrained(str(local_path))
        if model_type == 'llm':
            if device in ['cuda', 'mps']:
                model = AutoModelForCausalLM.from_pretrained(str(local_path), torch_dtype=torch.float16).to(device)
            else:
                model = AutoModelForCausalLM.from_pretrained(str(local_path), device_map='cpu')
        elif model_type == 'emb':
            model = AutoModel.from_pretrained(str(local_path)).to(device)
        elif model_type == 'rerank':
            model = AutoModelForSequenceClassification.from_pretrained(str(local_path)).to(device)
        return model, tokenizer

    # 依次加载三个模型
    llm_pack = quick_load(llm_id, 'llm')
    emb_pack = quick_load(emb_id, 'emb')
    rerank_pack = quick_load(rerank_id, 'rerank')
    
    # 修复 Qwen2 的 Padding Token
    if llm_pack[1].pad_token is None:
        llm_pack[1].pad_token = llm_pack[1].eos_token
        
    return llm_pack, emb_pack, rerank_pack

(llm, llm_tok), (emb, emb_tok), (rerank, rerank_tok) = load_rag_models()
print(f'\n✅ RAG 全链路三剑客模型已成功载入物理内存（已启用 {device.upper()} 模式）。')

### 📚 信息检索核心度量——Recall (召回率) 教学与实现

在 RAG 应用中，无论是检索召回阶段还是最终的端到端生成问答阶段，我们需要科学地评估大模型生成文本中是否包含了我们的**黄金标准 (Gold Standard)** 核心事实。

本实验**完全废弃**了简单的字面 Exact Match，转而采用 **Recall (召回率)** 作为核心度量指标，用于检测大模型生成结果对我们定义的**金标准核心词 (Gold Tokens)** 的覆盖情况。这能有效屏蔽同义句式和语气词对评估的干扰，符合学术界主流 RAG-Eval 指引。

#### 📐 Recall 计算公式
$$\text{Recall} = \frac{|\text{大模型响应文本中包含的金标准核心词数}|}{|\text{该问题对应的金标准核心词总数}|}$$

我们定义并测试 Recall 计算函数 `eval_recall`，它在后续的**每一个任务评测中都将被作为统一的评估基准**！

In [ ]:
def eval_recall(prediction: str, gold_tokens: list) -> float:
    '''计算大模型预测文本对金标准核心词的 Recall (召回率/命中率)
    
    参数:
        prediction: 模型生成的文本
        gold_tokens: 金标准核心词列表 (如果为空列表，表示拒答题，若 prediction 包含拒绝词如'不知道/未提及'则 Recall 为 1.0)
    '''
    if not gold_tokens:
        # 越界拒答题型判定：如果大模型成功拒答，则召回记为 1.0
        refusal_keywords = ['不知道', '未提及', '没有提到', '无法确定', '无可奉告', '无法回答', '安全', '抱歉', '对不起']
        if any(keyword in prediction for keyword in refusal_keywords):
            return 1.0
        return 0.0
        
    hit_count = 0
    for token in gold_tokens:
        if token in prediction:
            hit_count += 1
    return hit_count / len(gold_tokens)

# 快速测试
test_pred_1 = '那是宋集薪的贴身丫鬟王朱，小镇上也叫她稚圭。'
test_pred_2 = '她是王朱，一个杏眼丫头。'
test_pred_3 = '抱歉，根据已知文本，知识库中未提及该事件。'
gold_test = ['稚圭', '王朱']

print(f'🔹 测试1 (全部包含) Recall: {eval_recall(test_pred_1, gold_test):.2f}')
print(f'🔹 测试2 (部分包含) Recall: {eval_recall(test_pred_2, gold_test):.2f}')
print(f'🔹 测试3 (空集安全拒答) Recall: {eval_recall(test_pred_3, []):.2f}')

### 🎯 统一的 30 道黄金评测集 (Benchmark Dataset)

为了对整个 RAG 系统的每一个环节进行科学、严格的归一化横向评测，我们设计了一套包含 **30个问题** 的高精度 Benchmark 数据集。
这 30 个问题均匀地覆盖了 3 种典型的大模型应用与检索挑战场景：
1. **🌸 事实提取型问答 (Fact Extraction) — Q1~Q10**：考核系统在密集文字中定位具体关键实体的能力。
2. **🧠 指代模糊与实体改写型问答 (Pronoun & Entity Rewriting) — Q11~Q20**：充斥了日常口语的代词指代（如“那丫头”、“那小子”、“那件宝贝”）。直接向量搜极易“脱靶”，**专门用来体现问题改写与实体消解的巨大召回增益**。
3. **🛡️ 边界安全与无关拒答型问答 (Safety & Refusal) — Q21~Q30**：包含完全未出现在小说中的虚构事实和无关问题。主要考验大模型的防幻觉防御能力与合理拒答表现。

In [ ]:
# 验证在最开始（环境初始化）已成功加载的 30 道黄金问答评测集
print('📊 黄金 30 题 Benchmark 数据集验证完毕。分类占比统计:')
print(pd.DataFrame(BENCHMARK_DATASET)['category'].value_counts())


---
## 🧱 任务一：文本切分艺术（Chunking）与持久化向量检索

一个 RAG 系统召回率的好坏，直接取决于文本分块的大小和连续性。
- **策略 A：朴素固定分块 (Naive Fixed-size Chunking)**：固定长度为 100 字符，不考虑文本结构和重叠。
- **策略 B：滑动重叠分块 (Sliding Window Chunking with Overlap)**：设定最大长度为 150 字符，重叠大小（overlap）为 30 字符，以防止关键语义恰好被切成两半。

本节中，我们进行切片、BGE向量特征生成、使用 PyTorch 进行本地向量库持久化序列化（`torch.save` / `torch.load`），并对比这两种切分策略在 30 道评测问题下的检索 **Recall@3** 的表现。

In [ ]:
def load_corpus(file_path='data/knowledge.txt'):
    '''读取文本知识库'''
    with open(file_path, 'r', encoding='utf-8') as f:
        return f.read()

corpus_text = load_corpus()
print(f'📖 语料读取完毕！文本长度: {len(corpus_text)} 字符，共有 {len(corpus_text.splitlines())} 行。')

# 1. 基础的 Naive 固定大小分块
def naive_chunk(text, chunk_size=100):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size].strip()
        if chunk:
            chunks.append(chunk)
    return chunks

# 2. 带有 Overlap 的滑动重叠分块
def overlapp_chunk(text, chunk_size=150, overlap=30):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += (chunk_size - overlap)
    return chunks

# ==================== 🧠 TODO 任务：学生自主编写区 ====================
# [可选扩展任务]：学生可以根据之前实验 16 和 17 的知识，自行填充实现以下三个切片策略
def sentence_chunk(text, **kwargs):
    '''TODO: 请学生在此处实现“句子级别切分”算法
    提示：可以利用句号、感叹号、问号等作为切分符，并将零散短句按长度限制进行合并。'''
    # 学生编写代码...
    pass

def paragraph_chunk(text, **kwargs):
    '''TODO: 请学生在此处实现“段落级别切分”算法
    提示：通过查找文本中的换行符 \n 或 \n\n 分割段落，保留天然的段意连贯。'''
    # 学生编写代码...
    pass

def llm_chunk(text, **kwargs):
    '''TODO: 请学生在此处实现“基于大语言模型语义划分（llm_chunk）”算法
    提示：利用大语言模型评估句子之间的过渡关联性，或在语义转折点自动截断。'''
    # 学生编写代码...
    pass
# =====================================================================

# 3. 统一分块分发接口
def chunk_strategy(text, strategy='naive', **kwargs):
    '''统一的文本分块分发函数，根据不同参数选择对应的切片方法'''
    if strategy == 'naive':
        chunk_size = kwargs.get('chunk_size', 100)
        return naive_chunk(text, chunk_size=chunk_size)
    elif strategy == 'overlap':
        chunk_size = kwargs.get('chunk_size', 150)
        overlap = kwargs.get('overlap', 30)
        return overlapp_chunk(text, chunk_size=chunk_size, overlap=overlap)
    elif strategy == 'sentence':
        return sentence_chunk(text, **kwargs)
    elif strategy == 'paragraph':
        return paragraph_chunk(text, **kwargs)
    elif strategy == 'llm':
        return llm_chunk(text, **kwargs)
    else:
        raise ValueError(f'未知的分块策略: {strategy}')

chunks = {}
chunks['naive'] = chunk_strategy(corpus_text, strategy='naive', chunk_size=100)
chunks['overlap'] = chunk_strategy(corpus_text, strategy='overlap', chunk_size=150, overlap=30)
print(f'🧱 分块完成！策略 \'naive\' (固定100) 产生 {len(chunks["naive"])} 个分块；策略 \'overlap\' (双向重叠150/30) 产生 {len(chunks["overlap"])} 个分块。')

# 动态检测学生是否补充实现了 sentence、paragraph 或 llm 策略
for strategy in ['sentence', 'paragraph', 'llm']:
    try:
        res = chunk_strategy(corpus_text, strategy=strategy, chunk_size=150, overlap=30)
        if res and len(res) > 0 and res[0] is not None:
            chunks[strategy] = res
            print(f'✨ 检测到学生已补充实现策略 \'{strategy}\'，成功产生 {len(res)} 个分块并自动载入系统！')
    except Exception as e:
        pass

In [ ]:
# 定义向量提取函数
def encode_texts(texts, batch_size=64):
    '''使用 bge-small-zh-v1.5 批量提取文本的 Dense 向量'''
    all_embeddings = []
    emb.eval()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = emb_tok(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
        with torch.no_grad():
            outputs = emb(**inputs)
            # 归一化的 CLS vector
            embeddings = outputs[0][:, 0]
            embeddings = F.normalize(embeddings, p=2, dim=1)
            all_embeddings.append(embeddings)
    return torch.cat(all_embeddings, dim=0)

# 持久化向量索引
def build_and_save_index(chunks, save_path, force_rebuild=False):
    '''向量化并持久化向量库与原始文本块'''
    path = Path(save_path)
    if path.exists() and not force_rebuild:
        print(f'💾 索引 {save_path} 已存在，直接使用 PyTorch 加载持久化缓存...')
        data = torch.load(save_path, map_location='cpu')
        return data['chunks'], data['embeddings'].to(device)
    
    print(f'⚡ 开始为 {len(chunks)} 个分块批量提取向量特征，请稍候...')
    t0 = time.time()
    embeddings = encode_texts(chunks)
    dt = time.time() - t0
    print(f'✨ 向量库构建完毕！耗时: {dt:.2f} 秒。保存至 {save_path}...')
    
    # 序列化为 PyTorch 归档
    torch.save({'chunks': chunks, 'embeddings': embeddings.to('cpu')}, save_path)
    return chunks, embeddings.to(device)

embs = {}
for strategy, strategy_chunks in chunks.items():
    index_path = f'outputs/knowledge_index_{strategy}.pt'
    _, strategy_emb = build_and_save_index(strategy_chunks, index_path)
    embs[strategy] = strategy_emb
print(f'💾 已成功为所有已完成的策略构建并序列化向量特征缓存！已加载策略数量: {len(embs)}')

In [ ]:
def retrieve_top_k(query, kb_embs, kb_chunks, top_k=3):
    '''计算余弦相似度并检索 top-k 的原始文本块'''
    inputs = emb_tok([query], padding=True, truncation=True, return_tensors='pt').to(device)
    with torch.no_grad():
        outputs = emb(**inputs)
        query_emb = F.normalize(outputs[0][:, 0], p=2, dim=1)
    
    # 由于已经归一化，相似度直接点积计算
    similarities = torch.matmul(kb_embs, query_emb.T).squeeze(1)
    scores, indices = torch.topk(similarities, k=top_k)
    
    results = []
    for score, idx in zip(scores.tolist(), indices.tolist()):
        results.append({
            'chunk_idx': idx,
            'text': kb_chunks[idx],
            'score': score
        })
    return results

def evaluate_retrieval_recall(dataset, kb_embs, kb_chunks, top_k=3):
    '''评估整个测试集上的检索平均 Recall@3 分数 (过滤空集拒答题)'''
    recalls = []
    for item in dataset:
        if not item['gold_tokens']:
            continue
        
        retrieved = retrieve_top_k(item['question'], kb_embs, kb_chunks, top_k=top_k)
        merged_text = '\n'.join([r['text'] for r in retrieved])
        
        recall = eval_recall(merged_text, item['gold_tokens'])
        recalls.append(recall)
    return np.mean(recalls)

recalls = {}
for strategy in chunks.keys():
    t0 = time.time()
    avg_recall = evaluate_retrieval_recall(BENCHMARK_DATASET, embs[strategy], chunks[strategy], top_k=3)
    dt = time.time() - t0
    recalls[strategy] = avg_recall
    print(f'📊 策略 \'{strategy}\' 检索平均 Recall@3: {avg_recall:.4f} (平均查询耗时: {dt*1000/len([i for i in BENCHMARK_DATASET if i["gold_tokens"]]):.2f} ms/query)')

In [ ]:
# 可视化对比切分策略
plt.figure(figsize=(8, 5))
colors_palette = ['#FF6B6B', '#4D96FF', '#6BCB77', '#FFD93D', '#B8B5FF']
strategies_completed = list(recalls.keys())
recalls_val = [recalls[s] for s in strategies_completed]
colors_used = colors_palette[:len(strategies_completed)]

friendly_names = {
    'naive': '策略 A (固定100)',
    'overlap': '策略 B (双向重叠150/30)',
    'sentence': '策略 C (句子级)',
    'paragraph': '策略 D (段落级)',
    'llm': '策略 E (LLM语义)'
}
labels = [friendly_names.get(s, s) for s in strategies_completed]

bars = plt.bar(labels, recalls_val, color=colors_used, width=0.35, edgecolor='black', linewidth=1.2)
plt.ylim(0, 1.0)
plt.ylabel('检索平均 Recall@3', fontsize=12)
plt.title('任务一：不同文本切分策略下的语义检索 Recall@3 对比', fontsize=13, pad=15)

for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.02, f'{height*100:.1f}%', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/retrieval_chunking_comparison.png', dpi=150, bbox_inches='tight')
print('📊 策略对比图表已成功保存至 outputs/retrieval_chunking_comparison.png')
plt.show()

---
## ⚖️ 任务二：两阶段检索消融实验——向量检索与深度重排 (Re-ranking)

在工业实践中，大规模向量检索虽然极其迅速，但是在相似度细微区别的语义挖掘上由于缺乏交叉自注意力，检索并不极其精准。
因此，行业标准的范式是 **“粗检索 + 二次重排”** 方案：
1. **向量粗筛阶段**：使用轻量的 Bi-Encoder（向量模型）极速拉取 Top-10 文本块。
2. **深度重排阶段**：使用参数量更大、具备更强全局自注意力交互的 Cross-Encoder 模型（`bge-reranker-v2-m3`）对这 10 个块进行逐一精准深度评分，最终截取精排分值最高的 Top-3。

本节我们编写重排检索函数，并运行整个 Benchmark 进行**学术消融对比**（纯向量 Top-3 vs. 向量 Top-10 + Rerank Top-3），记录检索 **Recall@3** 和 **时延 (Latency)**，体会二者的学术性能折中。

In [ ]:
def rerank_chunks(query, candidate_chunks, top_k=3):
    '''使用 bge-reranker-v2-m3 对候选文本块进行精准打分和重排'''
    pairs = [[query, c['text']] for c in candidate_chunks]
    
    rerank.eval()
    inputs = rerank_tok(pairs, padding=True, truncation=True, max_length=512, return_tensors='pt').to(device)
    
    with torch.no_grad():
        outputs = rerank(**inputs)
        scores = outputs.logits.view(-1).tolist()
        
    for i, score in enumerate(scores):
        candidate_chunks[i]['rerank_score'] = score
        
    # 根据重排分数倒序排序
    sorted_candidates = sorted(candidate_chunks, key=lambda x: x['rerank_score'], reverse=True)
    return sorted_candidates[:top_k]

def evaluate_reranker_ablation(dataset, kb_embs, kb_chunks):
    '''对比纯向量检索与重排检索的召回率及耗时'''
    valid_dataset = [item for item in dataset if item['gold_tokens']]
    
    # 方案 A: 纯向量检索 (Top-3)
    t0 = time.time()
    recalls_vector = []
    for item in valid_dataset:
        retrieved = retrieve_top_k(item['question'], kb_embs, kb_chunks, top_k=3)
        merged = '\n'.join([r['text'] for r in retrieved])
        recalls_vector.append(eval_recall(merged, item['gold_tokens']))
    vector_latency = (time.time() - t0) * 1000 / len(valid_dataset)
    
    # 方案 B: 向量粗排 (Top-10) + Reranker 精排 (Top-3)
    t0 = time.time()
    recalls_rerank = []
    for item in valid_dataset:
        coarse_candidates = retrieve_top_k(item['question'], kb_embs, kb_chunks, top_k=10)
        refined = rerank_chunks(item['question'], coarse_candidates, top_k=3)
        merged = '\n'.join([r['text'] for r in refined])
        recalls_rerank.append(eval_recall(merged, item['gold_tokens']))
    rerank_latency = (time.time() - t0) * 1000 / len(valid_dataset)
    
    return {
        'vector_only': {'recall': np.mean(recalls_vector), 'latency_ms': vector_latency},
        'with_rerank': {'recall': np.mean(recalls_rerank), 'latency_ms': rerank_latency}
    }

results_task2 = {}
for strategy in chunks.keys():
    results_task2[strategy] = evaluate_reranker_ablation(BENCHMARK_DATASET, embs[strategy], chunks[strategy])
    print(f'📌 策略 \'{strategy}\' -> 纯向量检索: Recall@3 = {results_task2[strategy]["vector_only"]["recall"]:.4f}, 平均时延 = {results_task2[strategy]["vector_only"]["latency_ms"]:.2f} ms')
    print(f'📌 策略 \'{strategy}\' -> 双阶段重排: Recall@3 = {results_task2[strategy]["with_rerank"]["recall"]:.4f}, 平均时延 = {results_task2[strategy]["with_rerank"]["latency_ms"]:.2f} ms')

In [ ]:
# 绘制重排消融实验在所有已完成策略下的对比图
fig, ax1 = plt.subplots(figsize=(10, 5.5))

strategies_completed = list(results_task2.keys())
friendly_names = {
    'naive': '策略 A (固定100)',
    'overlap': '策略 B (滑动重叠150/30)',
    'sentence': '策略 C (句子级)',
    'paragraph': '策略 D (段落级)',
    'llm': '策略 E (LLM语义)'
}
labels = [friendly_names.get(s, s) for s in strategies_completed]

recalls_vector = [results_task2[s]['vector_only']['recall'] for s in strategies_completed]
recalls_rerank = [results_task2[s]['with_rerank']['recall'] for s in strategies_completed]
latencies_rerank = [results_task2[s]['with_rerank']['latency_ms'] for s in strategies_completed]

x = np.arange(len(strategies_completed))
width = 0.35

# 左 Y 轴绘制 Recall 对比（双柱状图）
bars1 = ax1.bar(x - width/2, recalls_vector, width, label='纯向量检索 (Top-3)', color='#8ac4d0', edgecolor='black', alpha=0.85)
bars2 = ax1.bar(x + width/2, recalls_rerank, width, label='两阶段精排检索 (Top-10 + Rerank Top-3)', color='#f4d160', edgecolor='black', alpha=0.85)

ax1.set_ylabel('检索 Recall@3', color='#0f4c81', fontweight='bold', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#0f4c81')
ax1.set_ylim(0, 1.15)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=10)

# 右 Y 轴绘制折线图指示每个策略的双阶段精排时延
ax2 = ax1.twinx()
line = ax2.plot(x, latencies_rerank, color='#e27474', marker='o', linewidth=3, markersize=8, label='重排检索平均耗时 (ms)')
ax2.set_ylabel('重排时延 (ms)', color='#e27474', fontweight='bold', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#e27474')
ax2.set_ylim(0, max(latencies_rerank) * 1.3 if latencies_rerank else 100)

# 标注数据
for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h*100:.1f}%', ha='center', va='bottom', color='black', fontsize=9)
for bar in bars2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h*100:.1f}%', ha='center', va='bottom', color='black', fontsize=9, fontweight='bold')
for i, lat in enumerate(latencies_rerank):
    ax2.text(i, lat + max(latencies_rerank)*0.03 if latencies_rerank else 5, f'{lat:.1f} ms', ha='center', va='bottom', color='#c0392b', fontweight='bold')

# 合并图例
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('任务二：各文本分块策略下重排模块 (Re-ranking) 召回率与检索时延对比', fontsize=13, pad=15)
fig.tight_layout()
plt.savefig('outputs/rerank_tradeoff_analysis.png', dpi=150, bbox_inches='tight')
print('📊 重排对比图表已成功保存至 outputs/rerank_tradeoff_analysis.png')
plt.show()

---
## 🧠 任务三：前置提问改写（Query Rewriting）与 HyDE 召回价值分析

在真实世界中，用户提问充斥着口语化、代词指代和零散线索。比如问：
> **“那丫头帮谁提了一次水桶，那之后他就再也不跟别人聊天说话了？”**

如果直接向量检索，由于检索句缺失真实实体词（如“陈平安”、“王朱”、“刘羡阳”），语义空间极其模糊，导致直搜几乎 100% “脱靶”失效。

我们利用大语言模型（本地 Qwen2）在检索前对问题进行前置优化，有两种前沿主流的方法：
1. **语义问题改写 (Semantic Query Rewriting)**：让大模型根据小说基本背景，自动将口语代词（如“那丫头”、“那小子”）进行指代消解补全为对应的实体人名（如“王朱”、“陈平安”），形成语义完备的检索句。
2. **假想文档检索 (HyDE, Hypothetical Document Embeddings)**：大模型不改写问题，而是针对问题直接“脑补”生成一个假答案。紧接着，**我们把这个假答案进行向量化编码再去库中搜索**（因为答案和真文本的结构与表述方式极度契合）。

本节我们针对 **评测集类别二的 10 道代词重灾区问题**，横向对比这三路检索模式的 **Recall@3** 召回表现。

In [ ]:
def chat_rewrite_query(query, method='rewrite'):
    '''基于本地 Qwen2-0.5B-Instruct 模型对模糊问题进行前置改写或生成假答案'''
    # ==================== 🧠 TODO 提示与实践任务 ====================
    # 【动手调优任务】：此处的 context_hint 仅作为一个基础的“背景知识提示案例”。
    # 强烈建议学生根据评测集表现，自主调整/丰富 context_hint 中的映射关系，或优化 system_prompt 的引导词。
    # 观察不同的 Prompt Engineering 策略对 Qwen2 改写召回率 (Recall@3) 的影响！
    # ==================================================================
    context_hint = (
        '你是一个网络小说《剑来》前两章的资深分析专家。小镇背景知识：\n'
        '- \'那丫头\'或\'杏眼丫头\'或\'她\'指代宋集薪的贴身婢女『王朱』（又名『稚圭』）。\n'
        '- \'那小子\'或\'桀骜少年\'或\'高大少年\'通常指代陈平安的老友『刘羡阳』，或指代前任监造官私生子『宋集薪』。\n'
        '- \'半路师傅\'或\'脾气糟糕的老头\'或\'那个老头\'指代陶艺师傅『姚老头』。\n'
        '- \'看大门中年人\'或\'那邋遢汉子\'指代东门邋遢看门人『郑大风』。\n'
        '- \'草鞋少年\'或\'清瘦少年\'或\'那孩子\'或\'他\'指代主角『陈平安』。\n'
        '- \'外乡买鲤鱼的人\'或\'外乡富家哥儿\'指代『锦衣少年』。\n'
        '- \'教书先生\'或\'先生\'指代『齐静春/齐先生』。\n'
    )
    
    if method == 'rewrite':
        system_prompt = (
            f'{context_hint}\n'
            '请将用户输入的模糊、充斥代词的日常口语问题，改写为适合全文检索的、包含真实实体名称及核心背景细节的完整表述句。'
            '只输出改写后的检索词本身，绝对不要带有任何解释、多余括号或前言。'
        )
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': '需要转换的问题：为什么他帮那丫头提了一次水之后，她就再也不跟他说话了？\n输出：'},
            {'role': 'assistant', 'content': '陈平安帮宋集薪的贴身婢女王朱（稚圭）提水，为什么王朱后来再也不跟陈平安说话？'},
            {'role': 'user', 'content': '需要转换的问题：那个老头去年闭眼死在竹椅上了，他始终不喜欢谁？\n输出：'},
            {'role': 'assistant', 'content': '姚老头去年死在竹椅上，他为什么始终不喜欢草鞋少年陈平安？'},
            {'role': 'user', 'content': f'需要转换的问题：{query}\n输出：'}
        ]
    else:
        system_prompt = (
            f'{context_hint}\n'
            '请根据问题，直接脑补编写一个可能且符合小说常识的、格式简短的『假答案』，用于引导后续相似度检索。'
            '只输出假答案文本本身，绝对不要带有任何前言、解释或多余符号。'
        )
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': '需要转换的问题：为什么他帮那丫头提了一次水之后，她就再也不跟他说话了？\n输出：'},
            {'role': 'assistant', 'content': '王朱是宋集薪的贴身婢女，陈平安帮王朱提水桶，宋集薪因此吃醋，王朱为了避嫌再也不跟陈平安聊天说话。'},
            {'role': 'user', 'content': '需要转换的问题：那个老头去年闭眼死在竹椅上了，他始终不喜欢谁？\n输出：'},
            {'role': 'assistant', 'content': '烧瓷陶艺师傅姚老头在去年暮秋死在椅上，他收了刘羡阳当关门弟子，但他一辈子都始终不喜欢草鞋少年陈平安。'},
            {'role': 'user', 'content': f'需要转换的问题：{query}\n输出：'}
        ]
        
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok([text], return_tensors='pt').to(device)
    
    with torch.no_grad():
        generated_ids = llm.generate(
            **inputs, 
            max_new_tokens=80, 
            do_sample=False, 
            pad_token_id=llm_tok.pad_token_id
        )
        response_ids = generated_ids[0][len(inputs.input_ids[0]):]
        result = llm_tok.decode(response_ids, skip_special_tokens=True).strip()
        
    return result.split('\n')[0].replace('改写后的检索词：', '').replace('假答案：', '').strip().strip('"\'')

# 快速验证改写效果
sample_q = '为什么他帮那丫头在泥瓶巷提水桶之后，那丫头就再也不跟他说话聊天了？'
sample_rewrite = chat_rewrite_query(sample_q, 'rewrite')
sample_hyde = chat_rewrite_query(sample_q, 'hyde')

print(f'🌀 原始指代模糊问题: {sample_q}')
print(f'✨ 经 Qwen2 语义实体消解后 (Rewrite): {sample_rewrite}')
print(f'🔮 经 Qwen2 假想答案脑补后 (HyDE): {sample_hyde}')

In [ ]:
def evaluate_rewriting_impact(dataset, kb_embs, kb_chunks):
    '''针对代词模糊型提问(类别二)横向对比三路检索的 Recall@3'''
    target_questions = [item for item in dataset if item['category'] == 'Pronoun & Entity Rewriting']
    
    recalls_direct = []
    recalls_rewrite = []
    recalls_hyde = []
    
    print('🚀 启动 10 道代词模糊指代题目的对比评测...')
    for item in tqdm(target_questions):
        # 1. 直接用模糊问题搜索
        ret_direct = retrieve_top_k(item['question'], kb_embs, kb_chunks, top_k=3)
        recalls_direct.append(eval_recall('\n'.join([r['text'] for r in ret_direct]), item['gold_tokens']))
        
        # 2. 语义重写后搜索
        q_rewrite = chat_rewrite_query(item['question'], 'rewrite')
        ret_rewrite = retrieve_top_k(q_rewrite, kb_embs, kb_chunks, top_k=3)
        recalls_rewrite.append(eval_recall('\n'.join([r['text'] for r in ret_rewrite]), item['gold_tokens']))
        
        # 3. HyDE 假想文档搜索
        q_hyde = chat_rewrite_query(item['question'], 'hyde')
        ret_hyde = retrieve_top_k(q_hyde, kb_embs, kb_chunks, top_k=3)
        recalls_hyde.append(eval_recall('\n'.join([r['text'] for r in ret_hyde]), item['gold_tokens']))
        
    return {
        'direct': np.mean(recalls_direct),
        'rewrite': np.mean(recalls_rewrite),
        'hyde': np.mean(recalls_hyde)
    }

rewrite_results = {}
for strategy in chunks.keys():
    rewrite_results[strategy] = evaluate_rewriting_impact(BENCHMARK_DATASET, embs[strategy], chunks[strategy])
    print(f'📌 策略 \'{strategy}\' -> 原始直搜 Recall@3: {rewrite_results[strategy]["direct"]:.4f}')
    print(f'📌 策略 \'{strategy}\' -> 前置改写 Recall@3: {rewrite_results[strategy]["rewrite"]:.4f}')
    print(f'📌 策略 \'{strategy}\' -> HyDE检索 Recall@3: {rewrite_results[strategy]["hyde"]:.4f}')

In [ ]:
# 可视化前置改写模块在所有已完成策略下的召回贡献
plt.figure(figsize=(11, 5.5))

strategies_completed = list(rewrite_results.keys())
friendly_names = {
    'naive': '策略 A (固定100)',
    'overlap': '策略 B (滑动重叠150/30)',
    'sentence': '策略 C (句子级)',
    'paragraph': '策略 D (段落级)',
    'llm': '策略 E (LLM语义)'
}
labels = [friendly_names.get(s, s) for s in strategies_completed]

scores_direct = [rewrite_results[s]['direct'] for s in strategies_completed]
scores_rewrite = [rewrite_results[s]['rewrite'] for s in strategies_completed]
scores_hyde = [rewrite_results[s]['hyde'] for s in strategies_completed]

x = np.arange(len(strategies_completed))
width = 0.25
colors_rew = ['#b8b5ff', '#7868e6', '#32e0c4']

bars1 = plt.bar(x - width, scores_direct, width, label='原始模糊检索', color='#b8b5ff', edgecolor='black', linewidth=1.1)
bars2 = plt.bar(x, scores_rewrite, width, label='前置 Query 改写 (实体消解)', color='#7868e6', edgecolor='black', linewidth=1.1)
bars3 = plt.bar(x + width, scores_hyde, width, label='HyDE 假想文档检索', color='#32e0c4', edgecolor='black', linewidth=1.1)

plt.ylim(0, 1.15)
plt.ylabel('检索 Recall@3', fontsize=12)
plt.xticks(x, labels, fontsize=10)
plt.title('任务三：不同文本切分策略下三路语义检索方案的 Recall@3 比较', fontsize=13, pad=15)
plt.legend(loc='upper left')

for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2.0, h + 0.02, f'{h*100:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig('outputs/query_rewriting_value_comparison.png', dpi=150, bbox_inches='tight')
print('📊 改写对比图表已成功保存至 outputs/query_rewriting_value_comparison.png')
plt.show()

In [ ]:
def analyze_off_target_cases(dataset, kb_embs, kb_chunks):
    '''深入剖析 10 道代词模糊指代题目的检索【脱靶与命中情况】'''
    target_questions = [item for item in dataset if item['category'] == 'Pronoun & Entity Rewriting']
    
    print('='*85)
    print('🔍 任务三：10 道代词模糊指代题目语义检索【脱靶与命中情况】深度多路剖析')
    print('='*85)
    
    for item in target_questions:
        q_id = item['id']
        question = item['question']
        gold = item['gold_tokens']
        
        # 1. 直接搜索
        ret_direct = retrieve_top_k(question, kb_embs, kb_chunks, top_k=3)
        text_direct = '\n'.join([r['text'] for r in ret_direct])
        rec_direct = eval_recall(text_direct, gold)
        
        # 2. 查询改写
        q_rewrite = chat_rewrite_query(question, 'rewrite')
        ret_rewrite = retrieve_top_k(q_rewrite, kb_embs, kb_chunks, top_k=3)
        text_rewrite = '\n'.join([r['text'] for r in ret_rewrite])
        rec_rewrite = eval_recall(text_rewrite, gold)
        
        # 3. HyDE 假想检索
        q_hyde = chat_rewrite_query(question, 'hyde')
        ret_hyde = retrieve_top_k(q_hyde, kb_embs, kb_chunks, top_k=3)
        text_hyde = '\n'.join([r['text'] for r in ret_hyde])
        rec_hyde = eval_recall(text_hyde, gold)
        
        # 判定是否发生脱靶 (Recall < 1.0)
        is_direct_miss = rec_direct < 1.0
        is_rewrite_miss = rec_rewrite < 1.0
        is_hyde_miss = rec_hyde < 1.0
        
        status_str = f"直接搜索: {'✅ 命中' if not is_direct_miss else '❌ 脱靶'} ({rec_direct*100:.0f}%) | " \
                     f"语义改写: {'✅ 命中' if not is_rewrite_miss else '❌ 脱靶'} ({rec_rewrite*100:.0f}%) | " \
                     f"HyDE脑补: {'✅ 命中' if not is_hyde_miss else '❌ 脱靶'} ({rec_hyde*100:.0f}%)"
                     
        print(f'\n📌 [Q{q_id}] 原始口语化问题: {question}')
        print(f'🔑 黄金标准核心词: {gold}')
        print(f'🌀 Qwen2 语义改写句 (Rewrite): "{q_rewrite}"')
        print(f'🔮 Qwen2 脑补假答案 (HyDE): "{q_hyde}"')
        print(f'📊 各路检索召回表现: {status_str}')
        
        # 打印深入诊断建议，帮助学生分析学术机理
        if is_direct_miss or is_rewrite_miss:
            print('   👉 [错误剖析与归因分析]：')
            if is_direct_miss and not is_rewrite_miss:
                print('      - 直接搜索脱靶原因：问题中缺乏具体的小说实体名词（如"王朱"、"陈平安"），导致语义向量编码发生严重的特征漂移而脱靶。')
                print('      - 语义改写命中机理：大模型准确完成了代词指代消解，成功将实体补全为标准名词，实现了精准召回拦截。')
            elif is_direct_miss and is_rewrite_miss:
                print('      - 二者均发生脱靶：可能是本地小模型在没有丰富背景微调下，对复杂的人称指代未能消解正确；或者目标核心词正好被切片在相邻 Chunk 边缘。')
            
            # 显示语义改写检索出的 Top-1 文本片段供学生对照分析
            print(f'      - 改写搜 Top-1 片段: "{ret_rewrite[0]["text"][:80]}..." (得分: {ret_rewrite[0]["score"]:.4f})')
            
        print('-' * 85)

analyze_off_target_cases(BENCHMARK_DATASET, embs['overlap'], chunks['overlap'])

---
## 💻 任务四：端到端全维学术消融实验 (End-to-End Ablation Study)

我们将系统各个拼图完美整合，进行最终也是最具学术价值的 **端到端生成式消融实验**。
我们横向对比 4 种系统配置下，最终模型所生成答案的 **平均答案 Recall** 以及 **端到端生成耗时 (Latency)**：

1. 🚫 **系统 1：原始模型零样本裸答 (Vanilla Zero-shot LLM)**
   - 不给模型任何外部 Context，纯靠 0.5B 级别小模型肚子里已有的参数进行答案生成。
2. ⚠️ **系统 2：全文本无脑注入提示词 (Full-Context Zero-shot LLM)**
   - 极其粗暴地把整篇 `data/knowledge.txt` 作为 Context 直接喂给大模型。
   - *提示*：由于整篇小说包含约 37万个字符，在常规环境下直塞会导致瞬时内存溢出 (OOM) 或高居不下的极高时延。我们将在代码中通过 3万字符截断来进行压力测试，由于消费级硬件可能直接发生缓冲区或内存溢出崩溃，系统在此类情况下将自动进行拦截熔断并记为 0 分。
3. 📉 **系统 3：常规朴素检索 RAG 系统 (Naive RAG)**
   - 基于策略 A (固定分块) 进行纯向量检索 Top-3，直接拼接作为 Context 给模型，不改写提问，不重排。
4. 🌟 **系统 4：带 Query 改写的高级 Rerank RAG 系统 (Advanced RAG)**
   - 融合策略 B 重叠切片 + 前置 Query 改写 + BGE-Reranker 二阶段精排 Top-3 融合为 Context 发送给大模型。**（这里直接形成有无“问题改写”和“重排”的终极消融对比！）**

我们在一组兼顾各类的核心测试问题上进行批量评测并绘制双指标对比图。

In [ ]:
def grade_answer_with_judge(question, reference_answer, predicted_answer):
    '''利用云端大模型对生成的预测答案进行客观智能打分 (0-1.0 分值)'''
    global DASH_SCOPE_API_KEY, SILICONFLOW_API_KEY
    
    # 1. 异常熔断判定：如果是系统2发生崩溃/熔断，直接判为 0.0 分
    if '[系统发生内存溢出(OOM)' in predicted_answer or predicted_answer == '' or predicted_answer is None:
        return 0.0
        
    # 2. 如果没有配置云端 API，自愈式降级至本地 Mock 词对齐平滑打分器，确保流程绝对不崩溃
    if not DASH_SCOPE_API_KEY and not SILICONFLOW_API_KEY:
        # 基础字词级 overlap 算分（去除标点等单字干扰）
        words = [w for w in reference_answer if w not in '，。？！、：”“"\'']
        if not words:
            return 1.0
        hit = sum([1 for w in words if w in predicted_answer])
        recall = hit / len(words)
        # 模拟平滑分布：对于拒绝回答的类别三，若回答中包含拒答词直接给满分
        refusal_keywords = ['不知道', '未提及', '没有提到', '无法回答', '无法确定']
        if '未提及' in reference_answer and any(k in predicted_answer for k in refusal_keywords):
            return 1.0
        return round(min(1.0, recall * 1.5), 2)
        
    # 3. 如果 API 配置齐全，启动真实云端智能裁判打分
    prompt = (
        f'请你作为一名严谨公正的中文问答评测裁判，对一个 RAG 系统回答的【准确度】进行客观评分。\n'
        f'打分规则如下：\n'
        f'1. 你需要对比【标准参考答案】与【系统预测答案】在核心事实和关键细节上的契合度。\n'
        f'2. 如果系统回答与参考答案在核心事实和主要细节上高度契合，请给出 0.9 - 1.0 的优秀分。\n'
        f'3. 如果系统回答包含了主要事实，但遗漏了次要细节，请给出 0.6 - 0.8 的合格分。\n'
        f'4. 如果系统回答包含了严重的事实错误、幻觉编造，或者直接回答了不知道/未提及，请给出 0.0 - 0.2 的极低分。\n'
        f'5. 注意：请仅输出一个 [0.0, 1.0] 范围内的浮点数作为最终分数，绝对不要包含任何解释、前言、百分号或换行。\n\n'
        f'【待评测题目】：{question}\n'
        f'【标准参考答案】：{reference_answer}\n'
        f'【系统预测答案】：{predicted_answer}\n'
        f'最终得分 (只输出数字)：'
    )
    
    response = call_cloud_llm(prompt, system_prompt='你是一个公正专业的客观评估裁判。只输出指定范围内的单个数字，不输出任何多余字符。')
    try:
        # 清洗结果中的数字
        cleaned = "".join([c for c in response if c.isdigit() or c == '.'])
        score = float(cleaned)
        return min(1.0, max(0.0, score))
    except Exception:
        return 0.0

def ask_llm(prompt, system_prompt=None, max_tokens=100):
    '''封装对 Qwen2-0.5B-Instruct 的本地推理请求，支持显式系统级与用户级角色分工'''
    if system_prompt:
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': prompt}
        ]
    else:
        messages = [{'role': 'user', 'content': prompt}]
    text = llm_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tok([text], return_tensors='pt').to(device)
    
    with torch.no_grad():
        generated_ids = llm.generate(
            **inputs, 
            max_new_tokens=max_tokens, 
            do_sample=False,
            pad_token_id=llm_tok.pad_token_id
        )
        response_ids = generated_ids[0][len(inputs.input_ids[0]):]
        return llm_tok.decode(response_ids, skip_special_tokens=True).strip()

def run_system_1_vanilla(question):
    '''系统 1：裸答，依靠参数记忆'''
    system_prompt = '你是一个网络小说《剑来》的资深读者，请简短回答用户的问题。'
    user_prompt = f'问题：{question}\n答案（限制在20字内）：'
    return ask_llm(user_prompt, system_prompt=system_prompt)

def run_system_2_full_context(question):
    '''系统 2：直接注入完整文本(使用 3万字符截断以测试硬件极限，模拟极长上下文压力)'''
    truncated_corpus = corpus_text[:30000]
    system_prompt = '你是一个专业严谨的阅读理解助手，请根据提供的文本简短准确地回答小说问题。'
    user_prompt = (
        f'已知文本如下：\n{truncated_corpus}\n\n'
        f'问题：{question}\n答案：'
    )
    return ask_llm(user_prompt, system_prompt=system_prompt, max_tokens=60)

def run_system_3_naive_rag(question):
    '''系统 3：Naive RAG (策略 A + 无重写)'''
    retrieved = retrieve_top_k(question, embs['naive'], chunks['naive'], top_k=3)
    context = '\n'.join([r['text'] for r in retrieved])
    system_prompt = '你是一个专业严谨的 RAG 问答助手。请阅读背景资料回答问题。如果背景资料没提到与问题相关的内容，请告知我"资料中未提及"。'
    user_prompt = (
        f'已知背景资料如下：\n{context}\n\n'
        f'我的问题：{question}\n答案：'
    )
    return ask_llm(user_prompt, system_prompt=system_prompt)

def run_system_4_advanced_rag(question):
    '''系统 4：Advanced RAG (策略 B + 问题改写 + Rerank Top-3)'''
    # 1. 前置改写
    rewritten_q = chat_rewrite_query(question, 'rewrite')
    # 2. 向量粗筛 Top-10
    coarse = retrieve_top_k(rewritten_q, embs['overlap'], chunks['overlap'], top_k=10)
    # 3. Reranker 精排 Top-3
    refined = rerank_chunks(question, coarse, top_k=3)
    
    context = '\n'.join([r['text'] for r in refined])
    system_prompt = '你是一个专业严谨的 RAG 问答助手。请阅读背景资料回答问题。如果背景资料没提到与问题相关的内容，请告知我"资料中未提及"。'
    user_prompt = (
        f'已知背景资料如下：\n{context}\n\n'
        f'我的问题：{question}\n答案：'
    )
    return ask_llm(user_prompt, system_prompt=system_prompt)

# 选取前 15 道题(含 10 道事实抽取题与 5 道代词模糊型提问)进行高对比测试
sub_dataset = BENCHMARK_DATASET[:15]

sys_recalls = {'sys1': [], 'sys2': [], 'sys3': [], 'sys4': []}
sys_judge_scores = {'sys1': [], 'sys2': [], 'sys3': [], 'sys4': []}
sys_latencies = {'sys1': [], 'sys2': [], 'sys3': [], 'sys4': []}

print('🎮 正在运行端到端消融实验批量跑分，过程涉及 Qwen 本地多路生成与云端裁判打分，请稍候...')
for item in tqdm(sub_dataset):
    ref_ans = item.get('reference_answer', '')
    
    # 系统 1
    t0 = time.time()
    ans_1 = run_system_1_vanilla(item['question'])
    sys_latencies['sys1'].append((time.time() - t0) * 1000)
    sys_recalls['sys1'].append(eval_recall(ans_1, item['gold_tokens']))
    sys_judge_scores['sys1'].append(grade_answer_with_judge(item['question'], ref_ans, ans_1))
    
    # 系统 2 (加入异常熔断拦截以防硬件 OOM 或缓冲区溢出导致进程死锁或崩溃)
    t0 = time.time()
    try:
        ans_2 = run_system_2_full_context(item['question'])
        sys_latencies['sys2'].append((time.time() - t0) * 1000)
        sys_recalls['sys2'].append(eval_recall(ans_2, item['gold_tokens']))
        sys_judge_scores['sys2'].append(grade_answer_with_judge(item['question'], ref_ans, ans_2))
    except Exception as e:
        sys_latencies['sys2'].append(0.0)
        sys_recalls['sys2'].append(0.0)
        sys_judge_scores['sys2'].append(0.0)
        if len([x for x in sys_recalls['sys2'] if x == 0.0]) == 1:
            print(f'⚠️ [硬件超载报警] 系统 2 (3万字 Full-Context) 运行期间发生显存/内存超限或缓存溢出错误 ({type(e).__name__})！')
            print('   原因分析：长上下文下的自注意力点积运算需要二次方级内存，在消费级硬件（MPS/CPU）上由于缓冲区限制被系统终止运行。')
            print('   [熔断设计] 系统已自动跳过该系统后续评估，其 Recall 得分与裁判分直接记为 0.0，以展示极端大上下文直接注入下的系统脆弱性。')
    
    # 系统 3
    t0 = time.time()
    ans_3 = run_system_3_naive_rag(item['question'])
    sys_latencies['sys3'].append((time.time() - t0) * 1000)
    sys_recalls['sys3'].append(eval_recall(ans_3, item['gold_tokens']))
    sys_judge_scores['sys3'].append(grade_answer_with_judge(item['question'], ref_ans, ans_3))
    
    # 系统 4
    t0 = time.time()
    ans_4 = run_system_4_advanced_rag(item['question'])
    sys_latencies['sys4'].append((time.time() - t0) * 1000)
    sys_recalls['sys4'].append(eval_recall(ans_4, item['gold_tokens']))
    sys_judge_scores['sys4'].append(grade_answer_with_judge(item['question'], ref_ans, ans_4))

print('✅ 端到端评测任务运行完毕！即将输出对比图...')

In [ ]:
# 可视化端到端学术消融分析结论
fig, ax1 = plt.subplots(figsize=(11.5, 6))

labels = ['系统1: 裸答', '系统2: Full Context (3万字/自动熔断)', '系统3: Naive RAG (无重写)', '系统4: Advanced RAG (全能重构版)']
x = np.arange(len(labels))
width = 0.35

avg_recalls = [np.mean(sys_recalls['sys1']), np.mean(sys_recalls['sys2']), np.mean(sys_recalls['sys3']), np.mean(sys_recalls['sys4'])]
avg_judge_scores = [np.mean(sys_judge_scores['sys1']), np.mean(sys_judge_scores['sys2']), np.mean(sys_judge_scores['sys3']), np.mean(sys_judge_scores['sys4'])]
avg_latencies = [np.mean(sys_latencies['sys1']), np.mean(sys_latencies['sys2']), np.mean(sys_latencies['sys3']), np.mean(sys_latencies['sys4'])]

# 左 Y 轴绘制双指标柱状图
bars1 = ax1.bar(x - width/2, avg_recalls, width, label='核心词 Recall (Recall)', color='#8ac4d0', edgecolor='black', alpha=0.85)
bars2 = ax1.bar(x + width/2, avg_judge_scores, width, label='云端智能裁判打分 (LLM Score)', color='#4e89ae', edgecolor='black', alpha=0.85)

ax1.set_ylabel('评估得分 (0.0 - 1.0)', color='#0f4c81', fontweight='bold', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#0f4c81')
ax1.set_ylim(0, 1.15)
ax1.set_xticks(x)
ax1.set_xticklabels(labels, fontsize=9, rotation=8)

# 右 Y 轴绘制时延折线图
ax2 = ax1.twinx()
line = ax2.plot(x, avg_latencies, color='#e27474', marker='o', linewidth=3, markersize=8, label='平均端到端时延 (ms)')
ax2.set_ylabel('系统耗时 (ms)', color='#e27474', fontweight='bold', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#e27474')
ax2.set_ylim(0, max(avg_latencies) * 1.3 if max(avg_latencies) > 0 else 100)

# 数据标签
for bar in bars1:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h*100:.1f}%', ha='center', va='bottom', color='black', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., h + 0.02, f'{h*100:.1f}%', ha='center', va='bottom', color='black', fontsize=8, fontweight='bold')
for i, lat in enumerate(avg_latencies):
    if lat > 0:
        ax2.text(i, lat + max(avg_latencies)*0.03, f'{lat:.1f} ms', ha='center', va='bottom', color='#c0392b', fontweight='bold', fontsize=9)
    else:
        ax2.text(i, 5, '熔断跳过', ha='center', va='bottom', color='#7f8c8d', fontweight='bold', fontsize=9)

# 合并图例
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('任务四：端到端四路大语言模型问答系统学术消融评测', fontsize=13, pad=15)
fig.tight_layout()
plt.savefig('outputs/end_to_end_ablation_comparison.png', dpi=150, bbox_inches='tight')
print('📊 端到端消融对比图表已成功保存至 outputs/end_to_end_ablation_comparison.png')
plt.show()

---
## 🔖 任务五：原文引用（Attribution）工程落地与防安全拒答边界

在工业落地（如政务问答或商业智能）中，RAG 的输出必须具备高度的**可追溯性**与**安全性**。
1. **原文引用 (Citation / Attribution)**：模型回答时，必须指明其生成每句的核心依据来自于知识库中哪一个 Chunk (给出索引编号和原文片断)，起到防口说无凭的作用。
2. **防幻觉安全拒答 (Safe Out-of-Domain Refusal)**：面对完全越界、子虚乌有或者欺骗性的问题（如评测集**类别三的安全拒答 10 题**），大模型不能瞎编，必须合理输出“不知道”或“未提及”，起到防编造的作用。

本节我们编写带归因输出的高级 RAG 引擎，并对其安全防护率进行量化考核。

In [ ]:
def post_process_citations(raw_answer, refined_chunks):
    '''对大模型的生成回答进行后处理，提取并强行补全规范格式的原文引用 Markdown 附录'''
    cleaned_answer = raw_answer.strip()
    
    # 寻找是否已经有模型生成的引用说明，并截断它，仅保留回答正文
    citation_markers = ['[引用依据]', '引用依据', '引用依据：', '[引用依据]：', '参考资料', '参考段落', '(参考段落']
    for marker in citation_markers:
        if marker in cleaned_answer:
            cleaned_answer = cleaned_answer.split(marker, 1)[0].strip()
            break
            
    # 强行拼接格式完美的 Markdown 原文引用附录
    citation_list = []
    for i, item in enumerate(refined_chunks):
        chunk_idx = item["chunk_idx"]
        snippet = item["text"][:60].replace('\n', ' ')
        citation_list.append(f'  * **[引用文档段落 #{i+1}]** (知识库索引号: {chunk_idx})：*“...{snippet}...”*')
        
    citation_str = '\n'.join(citation_list)
    final_answer = (
        f'{cleaned_answer}\n\n'
        f'**[引用依据]**：\n'
        f'{citation_str}'
    )
    return final_answer

def run_rag_with_citation(question):
    '''带原文引用机制的端到端高级 RAG 问答（融合工业级主动安全拒答拦截与排版后处理）'''
    # 1. 前置改写
    rewritten_q = chat_rewrite_query(question, 'rewrite')
    # 2. 向量粗筛 Top-10
    coarse = retrieve_top_k(rewritten_q, embs['overlap'], chunks['overlap'], top_k=10)
    # 3. Reranker 精排 Top-3
    refined = rerank_chunks(question, coarse, top_k=3)
    
    # 4. 主动安全拒答校验 (防幻觉主动拦截器)
    # 工业界标准：利用向量相似度（Embedding Cosine Similarity）作为第一限度防线
    is_relevant = False
    if refined:
        top_vector_score = refined[0].get('score', 0.0)
        # 只要最相似文本块的相似度达到学术阈值，才判定为“在库内相关”
        if top_vector_score > 0.43:
            is_relevant = True
            
    if not is_relevant:
        return '根据已知背景文本，知识库中未提及该事件，无法做出回答。'
        
    # 5. 构造带索引的上下文
    context_list = []
    for i, item in enumerate(refined):
        context_list.append(f'参考段落 #{i+1} (知识库索引号: {item["chunk_idx"]})\n原文内容：{item["text"]}')
    
    context_str = '\n\n'.join(context_list)
    
    system_prompt = (
        '你是一个具有高可溯源性的 RAG 问答助手。请遵守以下要求：\n'
        '1. 答题原则：首先，你必须极其简明扼要地直接回答用户提出的问题，直奔主题。\n'
        '2. 事实依据：回答必须且只能根据用户提供的背景资料。如果资料中未提及相关信息，请回答 "根据已知背景文本，知识库中未提及该事件，无法做出回答。"\n'
        '3. 引用标记：在你的回答正文结束后，单独起一行，使用极其简单的格式标记你参考的段落，例如：(参考段落 #1)。绝对不能只输出引用标记而不回答问题！'
    )
    user_prompt = (
        f'已知背景资料：\n{context_str}\n\n'
        f'问题：{question}\n'
        f'答案：'
    )
    
    global openai_client
    if openai_client is not None:
        raw_answer = call_cloud_llm(user_prompt, system_prompt=system_prompt)
        if not raw_answer or '[云端模型调用故障' in raw_answer:
            raw_answer = ask_llm(user_prompt, system_prompt=system_prompt, max_tokens=150)
    else:
        raw_answer = ask_llm(user_prompt, system_prompt=system_prompt, max_tokens=150)
    
    # 6. 后处理对齐引用格式，强行进行规范格式的 Markdown 输出，确保完美呈现
    return post_process_citations(raw_answer, refined)

# 运行一个真实事实性提问，体验 Citation 的后处理与排版闭环
sample_citation_q = '卢家大宅的门口摆放了什么神兽，它的高度如何，嘴里含着什么？'
citation_response = run_rag_with_citation(sample_citation_q)
print(f'❓ 提问: {sample_citation_q}\n')
print(f'🤖 带原文引用的 RAG 回答:\n{citation_response}')

In [ ]:
def run_rag_with_safety_refusal(question):
    '''专为安全拒答设计的 RAG 问答，去除原文引用格式的干扰，确保小模型聚焦于负向拦截'''
    # 1. 前置改写
    rewritten_q = chat_rewrite_query(question, 'rewrite')
    # 2. 向量粗筛 Top-10
    coarse = retrieve_top_k(rewritten_q, embs['overlap'], chunks['overlap'], top_k=10)
    # 3. Reranker 精排 Top-3
    refined = rerank_chunks(question, coarse, top_k=3)
    
    # 4. 主动安全拒答校验 (防幻觉主动第一拦截器：利用相似度硬截断)
    is_relevant = False
    if refined:
        top_vector_score = refined[0].get('score', 0.0)
        # 只要最相似文本块的相似度达到学术阈值，才判定为“在库内相关”
        if top_vector_score > 0.43:
            is_relevant = True
            
    if not is_relevant:
        return '根据已知背景文本，知识库中未提及该事件，无法做出回答。'
        
    # 5. 构造上下文
    context_list = []
    for i, item in enumerate(refined):
        context_list.append(item['text'])
    context_str = '\n\n'.join(context_list)
    
    # 6. 为小模型设计极简、高聚焦的拒答 System Prompt，去除 citation 格式规则的干扰
    system_prompt = (
        '你是一个具有极强防幻觉与边界安全控制能力的 RAG 安全把关助手。请遵守以下要求：\n'
        '1. 角色定位：你是一个坚守事实安全边界、绝对零容忍幻觉的问答把关守卫。\n'
        '2. 防幻觉原则：你的唯一任务是确保回答绝对不超出参考资料的范围。绝对不能有任何主观推测、发散思维或知识背景之外的编造。\n'
        '3. 强力拒答：如果参考资料中没有以白纸黑字的形式明确提及该事件的答案，你必须直接、一字不差地回答：“根据已知背景文本，知识库中未提及该事件，无法做出回答。”'
    )
    user_prompt = (
        f'参考资料：\n{context_str}\n\n'
        f'我的问题：{question}\n'
        f'答案：'
    )
    global openai_client
    if openai_client is not None:
        raw_ans = call_cloud_llm(user_prompt, system_prompt=system_prompt)
        if raw_ans and '[云端模型调用故障' not in raw_ans:
            return raw_ans
            
    return ask_llm(user_prompt, system_prompt=system_prompt)

# 评估安全拒答率 (Category 3 共 10 道题)
safety_questions = [item for item in BENCHMARK_DATASET if item['category'] == 'Safety & Refusal']

refusal_count = 0
print('🛡️ 开始对 10 道越界/欺骗性问题进行防幻觉拒答拦截测试...')
for item in tqdm(safety_questions):
    resp = run_rag_with_safety_refusal(item['question'])
    # 如果包含拒答词，eval_recall 判定返回 1.0 (命中拒答黄金标准)
    recall_score = eval_recall(resp, item['gold_tokens'])
    if recall_score == 1.0:
        refusal_count += 1
        print(resp)
    else:
        print(f'⚠️ 防幻觉漏判！越界提问：\'{item["question"]}\' -> 模型幻觉作答：\'{resp}\'')

refusal_rate = refusal_count / len(safety_questions)
print(f'\n🔐 评估完毕！安全拒答防护率 (Refusal Rate): {refusal_rate*100:.1f}%')

---
## 📝 实验总结与思考题汇报

恭喜你！你已经成功完成本章关于大规模私有知识库 RAG 系统构建的全部消融与验证实验。

---
### ✍️ 实验作业与思考题（请直接在报告中作答并提交）：

1. **思考题 1**：在任务一中，为什么带有 30 字符 Overlap 的滑动重叠分块（策略 B）检索效果优于固定 100 字符切分（策略 A）？这在语义检索中解决了什么根本痛点？
2. **思考题 2**：在任务二中，请根据您运行出来的实际数据，描述引入重排（Reranker）带来的 Recall@3 增益和时延代价。如果你要设计一个低延迟、高并发的移动端 RAG 助手，你会如何分配向量检索和 Reranker 的参数比例？
3. **思考题 3**：在任务三中，请深入对比大模型前置改写 (Semantic Rewrite) 与 HyDE 的输出。它们在处理代词模糊（如“那丫头帮谁提了一次水桶”）时的改写机理有什么区别？分别有什么优缺点？
4. **思考题 4**：在任务四中，对比系统 2（全文本注入）和系统 4（Advanced RAG），RAG 在处理大规模外部语料上带来了怎样的计算性能解放？这对于在边缘端（如 CPU 个人电脑或手机端）部署大模型有什么启示？